## Data Extraction

In [0]:
# Import the requests library to interact with HTTP APIs
# This will be used to fetch weather data from the Open-Meteo API
import requests

In [0]:
# Define the Open-Meteo API endpoint URL
url = "https://api.open-meteo.com/v1/forecast"

# Configure API request parameters:
parametros = {
    "latitude": -38.37,        # Latitude coordinate for the location (likely Argentina)
    "longitude": -60.28,       # Longitude coordinate for the location
    "hourly": "temperature_2m", # Request hourly temperature data at 2 meters above ground
    "past_days": 30,           # Retrieve data from the past 30 days
}

In [0]:
# Make HTTP GET request to the API with our defined parameters
response = requests.get(url, params=parametros)

# Check if the request was successful (HTTP status code 200 = OK)
if response.status_code == 200:
    # Parse the JSON response into a Python dictionary
    data = response.json()
    print("Conexion existosa")  # Connection successful
else:
    # Print error message if request failed
    print(f"error: {response.status_code}")

Conexion existosa


In [0]:
# Inspect the structure of the API response
# First, show the top-level keys in the response data
print(data.keys())
# Then, show the keys within the 'hourly' data section
print(data["hourly"].keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
dict_keys(['time', 'temperature_2m'])


## Data Proccesing

In [0]:
# Create the Bronze layer DataFrame (raw data ingestion)
# Combine timestamp and temperature lists into tuples (rows)
filas = list(zip(data["hourly"]["time"], data["hourly"]["temperature_2m"]))

# Define column names for our DataFrame
columnas = ["fecha_hora_texto", "temperatura"]  # datetime_text, temperature

# Create a Spark DataFrame from the raw API data
df_bronze = spark.createDataFrame(filas, schema=columnas)

# Display the bronze DataFrame to verify data ingestion
display(df_bronze)

fecha_hora_texto,temperatura
2026-02-15T00:00,21.7
2026-02-15T01:00,20.8
2026-02-15T02:00,19.8
2026-02-15T03:00,19.3
2026-02-15T04:00,18.7
2026-02-15T05:00,18.3
2026-02-15T06:00,18.1
2026-02-15T07:00,17.9
2026-02-15T08:00,17.4
2026-02-15T09:00,16.9


In [0]:
# Transform Bronze data to Silver layer (cleaned & standardized)
from pyspark.sql import functions as F

# Convert text datetime to Spark timestamp and drop the text column
df_silver = df_bronze.withColumn(
    "fecha_hora",
    F.to_timestamp("fecha_hora_texto", "yyyy-MM-dd'T'HH:mm")
).drop("fecha_hora_texto")

# Display the Silver DataFrame to verify timestamp conversion
display(df_silver)

temperatura,fecha_hora
21.7,2026-02-15T00:00:00.000Z
20.8,2026-02-15T01:00:00.000Z
19.8,2026-02-15T02:00:00.000Z
19.3,2026-02-15T03:00:00.000Z
18.7,2026-02-15T04:00:00.000Z
18.3,2026-02-15T05:00:00.000Z
18.1,2026-02-15T06:00:00.000Z
17.9,2026-02-15T07:00:00.000Z
17.4,2026-02-15T08:00:00.000Z
16.9,2026-02-15T09:00:00.000Z


In [0]:
# Save Silver layer DataFrame as a Delta table for downstream processing
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("clima_silver")

### Quick Sql for table creation

In [0]:
%sql
-- Create the Gold layer table using SQL for feature engineering
-- Adds hour, day of week, and lagged temperature as model signals
CREATE OR REPLACE TABLE clima_gold AS
SELECT 
    fecha_hora,
    hour(fecha_hora) as hora,             -- Extract hour of day
    dayofweek(fecha_hora) as dia_semana,  -- Extract day of the week
    temperatura as objetivo_temp,         -- Target value: temperature
    LAG(temperatura, 1) OVER (ORDER BY fecha_hora) as temp_h_anterior -- Previous hour temp
FROM clima_silver
WHERE temperatura IS NOT NULL

num_affected_rows,num_inserted_rows


## Machine Learning

In [0]:
# Prepare data for machine learning model
# Load Gold table, drop rows with nulls for reliable training
df_ml_input = spark.table("clima_gold")
df_ml_final = df_ml_input.dropna()
print(f"Registros disponibles: {df_ml_final.count()}")  # Report available records

Registros disponibles: 887


In [0]:
# Assemble features for ML model: hour, weekday, previous hour temp
from pyspark.ml.feature import VectorAssembler
pistas = ["hora", "dia_semana", "temp_h_anterior"]  # Model input columns
assembler = VectorAssembler(inputCols=pistas, outputCol="features")

df_entrenamiento = assembler.transform(df_ml_final)
# Display the features and target for visualization
display(df_entrenamiento.select("features", "objetivo_temp"))

features,objetivo_temp
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""1.0"",""1.0"",""21.7""]}",20.8
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""1.0"",""20.8""]}",19.8
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""1.0"",""19.8""]}",19.3
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""4.0"",""1.0"",""19.3""]}",18.7
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""5.0"",""1.0"",""18.7""]}",18.3
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""6.0"",""1.0"",""18.3""]}",18.1
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.0"",""1.0"",""18.1""]}",17.9
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.0"",""1.0"",""17.9""]}",17.4
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""9.0"",""1.0"",""17.4""]}",16.9
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""10.0"",""1.0"",""16.9""]}",16.8


In [0]:
# Train and evaluate a linear regression model to predict temperature
from pyspark.ml.regression import LinearRegression

# Split dataset into training and test sets (80/20 split)
train_df, test_df = df_entrenamiento.randomSplit([0.8, 0.2], seed=67)
lr = LinearRegression(featuresCol="features", labelCol="objetivo_temp")

# Fit model on training data
modelo_clima = lr.fit(train_df)

# Generate predictions on test data
predicciones = modelo_clima.transform(test_df)
# Display prediction results alongside features and true values
display(predicciones.select("features", "objetivo_temp", "prediction"))

features,objetivo_temp,prediction
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""3.0"",""1.0"",""19.8""]}",19.3,19.428628483544077
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""7.0"",""1.0"",""18.1""]}",17.9,18.05490876923706
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.0"",""1.0"",""17.9""]}",17.4,17.91973802382791
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""9.0"",""1.0"",""17.4""]}",16.9,17.50688836752862
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""19.0"",""1.0"",""27.5""]}",27.7,27.354896986006167
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""20.0"",""1.0"",""27.7""]}",27.5,27.589964788450544
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""22.0"",""1.0"",""25.3""]}",22.7,25.468430558364638
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""2.0"",""20.7""]}",18.9,20.098385956513496
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""2.0"",""2.0"",""17.6""]}",16.8,17.32893426768393
"{""type"":""1"",""size"":null,""indices"":null,""values"":[""8.0"",""2.0"",""15.3""]}",15.2,15.499753788631857


In [0]:
# Save ML predictions (timestamp, actual temp, predicted temp) as Delta table
predicciones.select("fecha_hora", "objetivo_temp", "prediction") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("clima_predicciones")